## Calculating AQI (Air Quality Index) in India with SNN approch

This notebook provides a workflow for calculating Air Quality Index. The formula is as per CPCB's official AQI Calculator: https://app.cpcbccr.com/ccr_docs/How_AQI_Calculated.pdf

I added SNN functionality, but most of the AQI caluclation remains the same

## Preparing data
The dataset used is hourly air quality data (2015 - 2020) from various measuring stations across India: https://www.kaggle.com/rohanrao/air-quality-data-in-india

We'll use one city (Thiruvananthapuram in Kerala) that has two stations and compare it with the actual AQI values present in the data at station, city, hour and day level to confirm the calculations are correct.


In [38]:
## importing packages
import numpy as np
import pandas as pd


In [39]:
## defining constants
PATH_STATION_HOUR = "./data/station_hour.csv"
PATH_STATIONS = "./data/stations.csv"

In [40]:
## importing data and subsetting the station
df = pd.read_csv(PATH_STATION_HOUR, parse_dates = ["Datetime"])
stations = pd.read_csv(PATH_STATIONS)

df = df.merge(stations, on = "StationId")

df.sort_values(["StationId", "Datetime"], inplace = True)
df.reset_index(drop = True, inplace = True)
print(f"Loaded {len(df):,} rows, {df.StationId.nunique()} stations")


/tmp/ipykernel_33109/3461417722.py:2: DtypeWarning: Columns (0: AQI_Bucket) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PATH_STATION_HOUR, parse_dates = ["Datetime"])


Loaded 2,589,083 rows, 110 stations


## Formula
![](https://i.imgur.com/vQR5Zy0.png)

* The AQI calculation uses 7 measures: **PM2.5, PM10, SO2, NOx, NH3, CO and O3**.
* For **PM2.5, PM10, SO2, NOx and NH3** the average value in last 24-hrs is used with the condition of having at least 16 values.
* For **CO and O3** the maximum value in last 8-hrs is used.
* Each measure is converted into a Sub-Index based on pre-defined groups.
* Sometimes measures are not available due to lack of measuring or lack of required data points.
* Final AQI is the maximum Sub-Index with the condition that at least one of PM2.5 and PM10 should be available and at least three out of the seven should be available.


In [41]:
df["PM10_24hr_avg"] = df.groupby("StationId")["PM10"].rolling(window = 24, min_periods = 16).mean().values
df["PM2.5_24hr_avg"] = df.groupby("StationId")["PM2.5"].rolling(window = 24, min_periods = 16).mean().values
df["SO2_24hr_avg"] = df.groupby("StationId")["SO2"].rolling(window = 24, min_periods = 16).mean().values
df["NOx_24hr_avg"] = df.groupby("StationId")["NOx"].rolling(window = 24, min_periods = 16).mean().values
df["NH3_24hr_avg"] = df.groupby("StationId")["NH3"].rolling(window = 24, min_periods = 16).mean().values
df["CO_8hr_max"] = df.groupby("StationId")["CO"].rolling(window = 8, min_periods = 1).max().values
df["O3_8hr_max"] = df.groupby("StationId")["O3"].rolling(window = 8, min_periods = 1).max().values


## PM2.5 (Particulate Matter 2.5-micrometer)
PM2.5 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:


In [42]:
## PM2.5 Sub-Index calculation
def get_PM25_subindex(x):
    if x <= 30:
        return x * 50 / 30
    elif x <= 60:
        return 50 + (x - 30) * 50 / 30
    elif x <= 90:
        return 100 + (x - 60) * 100 / 30
    elif x <= 120:
        return 200 + (x - 90) * 100 / 30
    elif x <= 250:
        return 300 + (x - 120) * 100 / 130
    elif x > 250:
        return 400 + (x - 250) * 100 / 130
    else:
        return 0

df["PM2.5_SubIndex"] = df["PM2.5_24hr_avg"].apply(lambda x: get_PM25_subindex(x))


## PM10 (Particulate Matter 10-micrometer)
PM10 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:


In [43]:
## PM10 Sub-Index calculation
def get_PM10_subindex(x):
    if x <= 50:
        return x
    elif x <= 100:
        return x
    elif x <= 250:
        return 100 + (x - 100) * 100 / 150
    elif x <= 350:
        return 200 + (x - 250)
    elif x <= 430:
        return 300 + (x - 350) * 100 / 80
    elif x > 430:
        return 400 + (x - 430) * 100 / 80
    else:
        return 0

df["PM10_SubIndex"] = df["PM10_24hr_avg"].apply(lambda x: get_PM10_subindex(x))


## SO2 (Sulphur Dioxide)
SO2 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:

In [44]:
## SO2 Sub-Index calculation
def get_SO2_subindex(x):
    if x <= 40:
        return x * 50 / 40
    elif x <= 80:
        return 50 + (x - 40) * 50 / 40
    elif x <= 380:
        return 100 + (x - 80) * 100 / 300
    elif x <= 800:
        return 200 + (x - 380) * 100 / 420
    elif x <= 1600:
        return 300 + (x - 800) * 100 / 800
    elif x > 1600:
        return 400 + (x - 1600) * 100 / 800
    else:
        return 0

df["SO2_SubIndex"] = df["SO2_24hr_avg"].apply(lambda x: get_SO2_subindex(x))


## NOx (Any Nitric x-oxide)
NOx is measured in ppb (parts per billion). The predefined groups are defined in the function below:


In [45]:
## NOx Sub-Index calculation
def get_NOx_subindex(x):
    if x <= 40:
        return x * 50 / 40
    elif x <= 80:
        return 50 + (x - 40) * 50 / 40
    elif x <= 180:
        return 100 + (x - 80) * 100 / 100
    elif x <= 280:
        return 200 + (x - 180) * 100 / 100
    elif x <= 400:
        return 300 + (x - 280) * 100 / 120
    elif x > 400:
        return 400 + (x - 400) * 100 / 120
    else:
        return 0

df["NOx_SubIndex"] = df["NOx_24hr_avg"].apply(lambda x: get_NOx_subindex(x))


## NH3 (Ammonia)
NH3 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:

In [46]:
## NH3 Sub-Index calculation
def get_NH3_subindex(x):
    if x <= 200:
        return x * 50 / 200
    elif x <= 400:
        return 50 + (x - 200) * 50 / 200
    elif x <= 800:
        return 100 + (x - 400) * 100 / 400
    elif x <= 1200:
        return 200 + (x - 800) * 100 / 400
    elif x <= 1800:
        return 300 + (x - 1200) * 100 / 600
    elif x > 1800:
        return 400 + (x - 1800) * 100 / 600
    else:
        return 0

df["NH3_SubIndex"] = df["NH3_24hr_avg"].apply(lambda x: get_NH3_subindex(x))


## CO (Carbon Monoxide)
CO is measured in mg / m3 (milligrams per cubic meter of air). The predefined groups are defined in the function below:

In [47]:
## CO Sub-Index calculation
def get_CO_subindex(x):
    if x <= 1:
        return x * 50 / 1
    elif x <= 2:
        return 50 + (x - 1) * 50 / 1
    elif x <= 10:
        return 100 + (x - 2) * 100 / 8
    elif x <= 17:
        return 200 + (x - 10) * 100 / 7
    elif x <= 34:
        return 300 + (x - 17) * 100 / 17
    elif x > 34:
        return 400 + (x - 34) * 100 / 17
    else:
        return 0

df["CO_SubIndex"] = df["CO_8hr_max"].apply(lambda x: get_CO_subindex(x))


## O3 (Ozone or Trioxygen)
O3 is measured in ug / m3 (micrograms per cubic meter of air). The predefined groups are defined in the function below:

In [48]:
## O3 Sub-Index calculation
def get_O3_subindex(x):
    if x <= 50:
        return x * 50 / 50
    elif x <= 100:
        return 50 + (x - 50) * 50 / 50
    elif x <= 168:
        return 100 + (x - 100) * 100 / 68
    elif x <= 208:
        return 200 + (x - 168) * 100 / 40
    elif x <= 748:
        return 300 + (x - 208) * 100 / 539
    elif x > 748:
        return 400 + (x - 400) * 100 / 539
    else:
        return 0

df["O3_SubIndex"] = df["O3_8hr_max"].apply(lambda x: get_O3_subindex(x))


## AQI
The final AQI is the maximum Sub-Index among the available sub-indices with the condition that at least one of PM2.5 and PM10 should be available and at least three out of the seven should be available.

There is no theoretical upper value of AQI but its rare to find values over 1000.

The pre-defined buckets of AQI are as follows:
![](https://i.imgur.com/XmnE0rT.png)


In [49]:
## AQI bucketing
def get_AQI_bucket(x):
    if x <= 50:
        return "Good"
    elif x <= 100:
        return "Satisfactory"
    elif x <= 200:
        return "Moderate"
    elif x <= 300:
        return "Poor"
    elif x <= 400:
        return "Very Poor"
    elif x > 400:
        return "Severe"
    else:
        return np.nan

df["Checks"] = (df["PM2.5_SubIndex"] > 0).astype(int) + \
                (df["PM10_SubIndex"] > 0).astype(int) + \
                (df["SO2_SubIndex"] > 0).astype(int) + \
                (df["NOx_SubIndex"] > 0).astype(int) + \
                (df["NH3_SubIndex"] > 0).astype(int) + \
                (df["CO_SubIndex"] > 0).astype(int) + \
                (df["O3_SubIndex"] > 0).astype(int)

df["AQI_calculated"] = round(df[["PM2.5_SubIndex", "PM10_SubIndex", "SO2_SubIndex", "NOx_SubIndex",
                                 "NH3_SubIndex", "CO_SubIndex", "O3_SubIndex"]].max(axis = 1))
df.loc[df["PM2.5_SubIndex"] + df["PM10_SubIndex"] <= 0, "AQI_calculated"] = np.nan
df.loc[df.Checks < 3, "AQI_calculated"] = np.nan

df["AQI_bucket_calculated"] = df["AQI_calculated"].apply(lambda x: get_AQI_bucket(x))
df[~df.AQI_calculated.isna()].head(13)


,StationId,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,PM2.5_SubIndex,PM10_SubIndex,SO2_SubIndex,NOx_SubIndex,NH3_SubIndex,CO_SubIndex,O3_SubIndex,Checks,AQI_calculated,AQI_bucket_calculated
16,AP001,2017-11-25 09:00:00,104.00,148.50,1.93,23.00,13.75,9.80,0.1,15.30,...,155.468750,112.562500,14.349219,15.366406,2.844844,5.0,125.911765,7,155.0,Moderate
17,AP001,2017-11-25 10:00:00,94.50,142.00,1.33,16.25,9.75,9.65,0.1,17.00,...,158.970588,113.470588,14.755147,15.179412,2.819412,5.0,153.279412,7,159.0,Moderate
18,AP001,2017-11-25 11:00:00,82.75,126.50,1.47,14.83,9.07,9.70,0.1,15.40,...,159.907407,113.703704,15.004861,14.965972,2.797500,5.0,173.411765,7,173.0,Moderate
19,AP001,2017-11-25 12:00:00,79.00,124.00,5.30,21.15,15.53,9.40,0.1,NaN,...,160.087719,113.824561,15.004861,15.200000,2.773947,5.0,183.529412,7,184.0,Moderate
20,AP001,2017-11-25 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,160.087719,113.824561,15.004861,15.200000,2.773947,5.0,183.529412,7,184.0,Moderate
21,AP001,2017-11-25 14:00:00,68.50,117.00,1.35,13.60,8.35,7.40,0.1,21.80,...,158.500000,113.700000,15.649342,14.961875,2.727750,5.0,190.735294,7,191.0,Moderate
22,AP001,2017-11-25 15:00:00,69.25,112.25,1.52,11.80,7.55,9.25,0.1,21.38,...,157.182540,113.436508,16.203125,14.698810,2.707976,5.0,190.735294,7,191.0,Moderate
23,AP001,2017-11-25 16:00:00,70.00,107.00,2.80,30.33,18.40,6.15,0.1,18.90,...,156.098485,113.037879,16.556548,15.076136,2.654773,5.0,190.735294,7,191.0,Moderate
24,AP001,2017-11-25 17:00:00,72.75,120.25,1.50,26.72,15.45,10.78,0.1,16.03,...,157.954545,113.712121,16.805357,14.917045,2.680682,5.0,190.735294,7,191.0,Moderate
25,AP001,2017-11-25 18:00:00,81.50,134.75,1.10,18.78,10.88,14.73,0.1,12.93,...,160.378788,114.424242,16.791071,14.678977,2.737045,5.0,190.735294,7,191.0,Moderate


In [50]:
df[~df.AQI_calculated.isna()].AQI_bucket_calculated.value_counts()

AQI_bucket_calculated
Moderate        674415
Satisfactory    529974
Very Poor       301128
Poor            239914
Good            152073
Severe          120468
Name: count, dtype: int64

Matches perfectly. In case of any discrepancy or bug or issue, feel free to comment here or share on the [dataset page](https://www.kaggle.com/rohanrao/air-quality-data-in-india).

# SNN Part

I kept most of the AQI calcualtion as in the tutorial (except using all stations and local data paths)

Below we turn the 7 raw pollutant readings into features, then build a classifier using spikes. The main chnages are:

 - **Input encoding** floats with `spikegen.rate` to binary spike trains
 - Replacing `ReLU` with `snn.Leaky` (LIF neurons) as **activation function**
 - **Forward pass** is nowa  loop over timesteps that accumulates output spikes instead of default single pass

In [51]:
## SNN imports (the only new dependencies)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

import snntorch as snn
from snntorch import spikegen

### Feature preparation
7 raw pollutant columns as features and the calculated AQI bucket as the target label. Rows with missing AQI are dropped. Features are Z-score normalised then clipped to [0, 1] so they can be treated as spike probabilities

In [52]:
FEATURES = ["PM2.5", "PM10", "SO2", "NOx", "NH3", "CO", "O3"]
BUCKET_TO_INT = {"Good": 0, "Satisfactory": 1, "Moderate": 2, "Poor": 3, "Very Poor": 4, "Severe": 5}
INT_TO_BUCKET = {v: k for k, v in BUCKET_TO_INT.items()}
NUM_STEPS = 10

snn_df = df[FEATURES + ["AQI_bucket_calculated"]].copy()
snn_df[FEATURES] = snn_df[FEATURES].ffill()
snn_df.dropna(subset=["AQI_bucket_calculated"], inplace=True)
snn_df = snn_df[snn_df["AQI_bucket_calculated"] != "0"]

snn_df["label"] = snn_df["AQI_bucket_calculated"].map(BUCKET_TO_INT)
snn_df.dropna(subset=["label"], inplace=True)
snn_df["label"] = snn_df["label"].astype(int)

print(f"Samples with valid AQI bucket: {len(snn_df):,}")
print(snn_df["AQI_bucket_calculated"].value_counts())

Samples with valid AQI bucket: 2,017,972
AQI_bucket_calculated
Moderate        674415
Satisfactory    529974
Very Poor       301128
Poor            239914
Good            152073
Severe          120468
Name: count, dtype: int64


## Temporal split 80% train and 20% testing, normalization + loading data

In [53]:
## Temporal train/test split (80/20) and normalisation
split = int(len(snn_df) * 0.8)
train_df = snn_df.iloc[:split]
test_df  = snn_df.iloc[split:]

mean = train_df[FEATURES].mean()
std  = train_df[FEATURES].std().replace(0, 1)

def normalize(part):
    X = ((part[FEATURES] - mean) / std).clip(0, 1).values.astype(np.float32)
    y = part["label"].values
    return X, y

X_train, y_train = normalize(train_df)
X_test,  y_test  = normalize(test_df)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
                          batch_size=256, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.tensor(X_test),  torch.tensor(y_test)),
                          batch_size=512)

print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

Train: 1,614,377  Test: 403,595


### SNN-related changes
#### Spike Encoding
Converting float samples to Bernoulli firing probability for spike readings [0,1] using `spikegen.rate`.
#### LIF neurons
Replacing ReLU used in the standard MLP with LIF neurons that maintain a membrane potential across timesteps and emit a binary spike when a threshold is crossed using `snn.Leaky`.
#### Timestep loop + spike-count readout
Instead of one forward pass, we loop over each timestep, feeding one "frame" of spikes through the network and accumulating output spikes. The class with the most output spikes wins.

In [54]:
class AQI_SNN(nn.Module):
    def __init__(self, num_features=7, num_classes=6):
        super().__init__()
        self.fc1     = nn.Linear(num_features, 128)
        self.fc2     = nn.Linear(128, 64)
        self.fc_out  = nn.Linear(64, num_classes)

        # SNN Change 2
        self.lif1    = snn.Leaky(beta=0.95)
        self.lif2    = snn.Leaky(beta=0.90)
        self.lif_out = snn.Leaky(beta=0.85)

    # spikes: [NUM_STEPS, batch, 7]
    def forward(self, spikes):
        T = spikes.shape[0]
        mem1   = self.lif1.init_leaky()
        mem2   = self.lif2.init_leaky()
        mem_out = self.lif_out.init_leaky()
        out_rec = torch.zeros(spikes.shape[1], self.fc_out.out_features, device=spikes.device)

        # SNN Change 3
        for t in range(T):
            x = spikes[t]
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            cur_out = self.fc_out(spk2)
            spk_out, mem_out = self.lif_out(cur_out, mem_out)
            out_rec += spk_out

        return out_rec

In [55]:
# Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AQI_SNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # SNN Change 1
        spike_data = spikegen.rate(X_batch, num_steps=NUM_STEPS)

        out = model(spike_data)
        loss = criterion(out, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)
        correct += (out.argmax(1) == y_batch).sum().item()
        total += len(y_batch)

    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={total_loss/total:.4f}  acc={correct/total:.3f}")

Epoch  1/5  loss=1.1937  acc=0.466
Epoch  2/5  loss=1.1791  acc=0.468
Epoch  3/5  loss=1.1745  acc=0.472
Epoch  4/5  loss=1.1724  acc=0.472
Epoch  5/5  loss=1.1703  acc=0.473


In [56]:
### Evaluation: spike-encoding + forward pass
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        spike_data = spikegen.rate(X_batch, num_steps=NUM_STEPS)
        out = model(spike_data)
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(y_batch.tolist())

present = sorted(set(all_labels) | set(all_preds))
target_names = [INT_TO_BUCKET[i] for i in present]

acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\nTest accuracy: {acc:.3f}")
print(f"Baseline (most-frequent class): {max(all_labels.count(c) for c in present) / len(all_labels):.3f}\n")
print(classification_report(all_labels, all_preds, labels=present, target_names=target_names))
print("Confusion matrix:")
print(confusion_matrix(all_labels, all_preds, labels=present))


Test accuracy: 0.491
Baseline (most-frequent class): 0.362

              precision    recall  f1-score   support

        Good       0.13      0.01      0.03     44087
Satisfactory       0.50      0.80      0.61    146012
    Moderate       0.47      0.42      0.44    121940
        Poor       0.32      0.07      0.11     34001
   Very Poor       0.58      0.58      0.58     46286
      Severe       0.24      0.01      0.02     11269

    accuracy                           0.49    403595
   macro avg       0.37      0.32      0.30    403595
weighted avg       0.44      0.49      0.43    403595

Confusion matrix:
[[   630  41471   1708     64    182     32]
 [  1634 117358  25857    230    926      7]
 [  1637  64973  50653   1262   3375     40]
 [   374   7771  17584   2278   5953     41]
 [   379   4390  11434   2960  26956    167]
 [   245    748    780    283   9120     93]]
